# Retail Data Analytics and Data Wrangling

## Imports

In [0]:
%scala
import org.apache.spark.sql.functions._
import spark.implicits._
import org.apache.spark.sql.expressions.Window
import java.sql.Date

import org.apache.spark.sql.functions._
import spark.implicits._
import org.apache.spark.sql.expressions.Window
import java.sql.Date

## Data Preparation

In [0]:
%scala
// File location and type
val f1_location = "/FileStore/tables/retail-1.csv"
val file_type = "csv"

var retail_df = spark.read
  .format(file_type)
  .option("inferSchema", "true")
  .option("header", "true")
  .option("sep", ",")
  .load(f1_location)

// Add total_amount column
retail_df = retail_df.withColumn("total_amount", round(col("quantity") * col("unit_price"), 2))

// Show only the columns we care about
retail_df.select("invoice_no", "quantity", "unit_price", "total_amount").show(10)

+----------+--------+----------+------------+
invoice_no|quantity|unit_price|total_amount|
+----------+--------+----------+------------+
 489434| 12| 6.95| 83.4|
 489434| 12| 6.75| 81.0|
 489434| 12| 6.75| 81.0|
 489434| 48| 2.1| 100.8|
 489434| 24| 1.25| 30.0|
 489434| 24| 1.65| 39.6|
 489434| 24| 1.25| 30.0|
 489434| 10| 5.95| 59.5|
 489435| 12| 2.55| 30.6|
 489435| 12| 3.75| 45.0|
+----------+--------+----------+------------+
only showing top 10 rows

f1_location: String = /FileStore/tables/retail-1.csv
file_type: String = csv
retail_df: org.apache.spark.sql.DataFrame = [invoice_no: string, stock_code: string ... 7 more fields]
retail_df: org.apache.spark.sql.DataFrame = [invoice_no: string, stock_code: string ... 7 more fields]

In [0]:
%scala
retail_df.printSchema()

root
-- invoice_no: string (nullable = true)
-- stock_code: string (nullable = true)
-- description: string (nullable = true)
-- quantity: integer (nullable = true)
-- invoice_date: timestamp (nullable = true)
-- unit_price: double (nullable = true)
-- customer_id: integer (nullable = true)
-- country: string (nullable = true)
-- total_amount: double (nullable = true)

## Total Invoice Amount Distribution

In [0]:
%scala
// filtered_df = retail_df[retail_df['invoice_no'].str.isdigit() & (retail_df['total_amount'] > 0)]
// invoice_total_df = filtered_df.groupby('invoice_no')['total_amount'].sum()
val isDigitUDF = udf((s: String) => s != null && s.forall(_.isDigit))
val filtered_df = retail_df
  .filter(isDigitUDF(col("invoice_no")) && col("total_amount") > 0)

val invoice_total_df = filtered_df
  .groupBy("invoice_no") 
  .agg(sum("total_amount").alias("total_amount"))


isDigitUDF: org.apache.spark.sql.expressions.UserDefinedFunction = SparkUserDefinedFunction($Lambda$9061/335714473@28cb3505,BooleanType,List(Some(class[value[0]: string])),Some(class[value[0]: boolean]),None,false,true)
filtered_df: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [invoice_no: string, stock_code: string ... 7 more fields]
invoice_total_df: org.apache.spark.sql.DataFrame = [invoice_no: string, total_amount: double]

In [0]:
%scala
val statistics = invoice_total_df
  .agg(
    min("total_amount").alias("Min"),
    max("total_amount").alias("Max"),
    avg("total_amount").alias("Mean"),
    expr("percentile_approx(total_amount, 0.5)").alias("Median")
  )

val stats = statistics.collect()(0)

// mode is a bit more complicated to get as there is no defined function within agg
val mode = invoice_total_df
  .groupBy("total_amount")
  .count()
  .orderBy(desc("count"))
  .limit(1)
  .collect()(0)

println(s"Minimum: ${stats.getAs[Double]("Min")}")
println(s"Maximum: ${stats.getAs[Double]("Max")}")
println(s"Mean: ${stats.getAs[Double]("Mean")}")
println(s"Median: ${stats.getAs[Double]("Median")}")
println(s"Mode: ${mode.getAs[Double]("total_amount")}")

Minimum: 0.19
Maximum: 168469.6
Mean: 523.0445263998419
Median: 304.29999999999995
Mode: 15.0
statistics: org.apache.spark.sql.DataFrame = [Min: double, Max: double ... 2 more fields]
stats: org.apache.spark.sql.Row = [0.19,168469.6,523.0445263998419,304.29999999999995]
mode: org.apache.spark.sql.Row = [15.0,119]

In [0]:
%scala
val histogram_df = invoice_total_df.select("total_amount")
display(histogram_df)

total_amount
192.0
303.2
155.05999999999997
118.75
275.95
6711.0
1335.92
2507.06
48.96
199.3


In [0]:
%scala
val quant_val = invoice_total_df.stat.approxQuantile("total_amount", Array(0.85), 0.01)(0)

val invoice_total_df_agg = invoice_total_df
  .groupBy("invoice_no")
  .agg(sum("total_amount").alias("total_amount"))

val invoice_total_outliers_df = invoice_total_df
  .filter(col("total_amount") <= quant_val)

quant_val: Double = 691.4000000000002
invoice_total_df_agg: org.apache.spark.sql.DataFrame = [invoice_no: string, total_amount: double]
invoice_total_outliers_df: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [invoice_no: string, total_amount: double]

In [0]:
%scala
val statistics = invoice_total_outliers_df
  .agg(
    min("total_amount").alias("Min"),
    max("total_amount").alias("Max"),
    avg("total_amount").alias("Mean"),
    expr("percentile_approx(total_amount, 0.5)").alias("Median")
  )

val stats = statistics.collect()(0)

// mode is a bit more complicated to get as there is no defined function within agg
val mode = invoice_total_outliers_df
  .groupBy("total_amount")
  .count()
  .orderBy(desc("count"))
  .limit(1)
  .collect()(0)

println(s"Minimum: ${stats.getAs[Double]("Min")}")
println(s"Maximum: ${stats.getAs[Double]("Max")}")
println(s"Mean: ${stats.getAs[Double]("Mean")}")
println(s"Median: ${stats.getAs[Double]("Median")}")
println(s"Mode: ${mode.getAs[Double]("total_amount")}")

Minimum: 0.19
Maximum: 691.4000000000002
Mean: 266.71517457395765
Median: 252.8
Mode: 15.0
statistics: org.apache.spark.sql.DataFrame = [Min: double, Max: double ... 2 more fields]
stats: org.apache.spark.sql.Row = [0.19,691.4000000000002,266.71517457395765,252.8]
mode: org.apache.spark.sql.Row = [15.0,119]

In [0]:
%scala
val histogram_df2 = invoice_total_outliers_df.select("total_amount")
display(histogram_df2)

total_amount
192.0
303.2
155.05999999999997
118.75
275.95
48.96
199.3
188.82999999999998
291.14000000000004
312.59


## Monthly Placed and Canceled Order

In [0]:
%scala
retail_df = retail_df.withColumn("invoice_date", date_format(col("invoice_date"), "yyyyMM"))

retail_df: org.apache.spark.sql.DataFrame = [invoice_no: string, stock_code: string ... 7 more fields]

In [0]:
%scala
var filtered_cancels = retail_df
  .filter(col("total_amount") < 0) 
  .dropDuplicates("invoice_no")

val cancels = filtered_cancels
  .groupBy("invoice_date") 
  .agg(count("invoice_no").alias("cancels"))

val orders = retail_df
  .dropDuplicates("invoice_no")
  .groupBy("invoice_date") 
  .agg(count("invoice_no").alias("placements"))

var monthly_orders = orders
  .join(cancels, "invoice_date")  // Correct join syntax
  .withColumn("Placement", col("placements") - (lit(2) * col("cancels")))
  .select("invoice_date", "Placement", "cancels")
  .orderBy("invoice_date")

monthly_orders = monthly_orders.withColumnRenamed("cancels", "Cancellation")

// cancels.show(5)
// orders.show(5)
monthly_orders.show(5)

+------------+---------+------------+
invoice_date|Placement|Cancellation|
+------------+---------+------------+
 200912| 1528| 401|
 201001| 1033| 300|
 201002| 1491| 239|
 201003| 1553| 407|
 201004| 1282| 305|
+------------+---------+------------+
only showing top 5 rows

filtered_cancels: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [invoice_no: string, stock_code: string ... 7 more fields]
cancels: org.apache.spark.sql.DataFrame = [invoice_date: string, cancels: bigint]
orders: org.apache.spark.sql.DataFrame = [invoice_date: string, placements: bigint]
monthly_orders: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [invoice_date: string, Placement: bigint ... 1 more field]
monthly_orders: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [invoice_date: string, Placement: bigint ... 1 more field]

In [0]:
%scala
display(monthly_orders)

invoice_date,Placement,Cancellation
200912,1528,401
201001,1033,300
201002,1491,239
201003,1553,407
201004,1282,305
201005,1604,407
201006,1502,357
201007,1327,345
201008,1331,273
201009,1633,371


## Montly Sales

In [0]:
%scala
val monthly_sales = retail_df
    .filter(col("invoice_no").rlike("^[0-9]+$")) 
    .groupBy("invoice_date")
    .agg(sum("total_amount").alias("total_sales"))
    .orderBy("invoice_date")

monthly_sales: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [invoice_date: string, total_sales: double]

In [0]:
%scala
display(monthly_sales)

invoice_date,total_sales
200912,825685.7600000115
201001,652708.4999999943
201002,553339.7300000008
201003,833570.129999991
201004,681528.9899999797
201005,659858.8599999907
201006,752270.1499999794
201007,650712.9400000102
201008,697274.9099999849
201009,924333.0099999603


## Monthly Sales Growth

In [0]:
%scala
val window_spec = Window.orderBy(col("invoice_date"))

val monthly_growth = monthly_sales.withColumn(
  "previous_month_sales", lag(col("total_sales"), 1).over(window_spec)
).withColumn(
  "sales_growth", ((col("total_sales") - col("previous_month_sales")) / col("previous_month_sales")) * 100
)

// monthly_growth.show(5)

window_spec: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@25f6c8ab
monthly_growth: org.apache.spark.sql.DataFrame = [invoice_date: string, total_sales: double ... 2 more fields]

In [0]:
%scala
val monthly_active = retail_df
  .groupBy("invoice_date")
  .agg(countDistinct("customer_id").alias("active_customers"))
  .orderBy("invoice_date")

monthly_active.show(5)

+------------+----------------+
invoice_date|active_customers|
+------------+----------------+
 200912| 1045|
 201001| 786|
 201002| 807|
 201003| 1111|
 201004| 998|
+------------+----------------+
only showing top 5 rows

monthly_active: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [invoice_date: string, active_customers: bigint]

In [0]:
%scala
display(monthly_active)

invoice_date,active_customers
200912,1045
201001,786
201002,807
201003,1111
201004,998
201005,1062
201006,1095
201007,988
201008,964
201009,1202


## New and Existing Users

In [0]:
%scala
retail_df = retail_df.withColumn("invoice_date", date_format(col("invoice_date"), "yyyyMM"))

retail_df: org.apache.spark.sql.DataFrame = [invoice_no: string, stock_code: string ... 7 more fields]

In [0]:
%scala
val window_spec = Window.partitionBy("customer_id")
var temp_df = retail_df.withColumn("first_purchase_month", min(col("invoice_date")).over(window_spec))

var new_users = temp_df.filter(col("first_purchase_month") === col("invoice_date"))

val new_users_count = new_users
  .groupBy("invoice_date") 
  .agg(countDistinct("customer_id").alias("new_users_count")) 
  .orderBy("invoice_date")

val previous_users = temp_df.filter(col("first_purchase_month") < col("invoice_date"))

val previous_users_count = previous_users
  .groupBy("invoice_date") 
  .agg(countDistinct("customer_id").alias("previous_users_count"))
  .orderBy("invoice_date")

val new_users_count_df = new_users_count.withColumnRenamed("new_users_count", "New Users")
val previous_users_count_df = previous_users_count.withColumnRenamed("previous_users_count", "Previous Users")

val mergedData = new_users_count_df.join(previous_users_count_df, Seq("invoice_date"), "outer")
  .na.fill(0)
  .orderBy("invoice_date")

display(mergedData)

invoice_date,New Users,Previous Users
+20091201,1045,0
+20100101,394,392
+20100201,363,444
+20100301,436,675
+20100401,291,707
+20100501,254,808
+20100601,269,826
+20100701,183,805
+20100801,158,806
+20100901,242,960


## Finding RFM

In [0]:
%scala // not the most correct yyyyMMdd would be better but made mistake in pandas and have to keep consistent for comparison
retail_df = retail_df.withColumn("invoice_date", date_format(col("invoice_date"), "yyyyMM"))

retail_df: org.apache.spark.sql.DataFrame = [invoice_no: string, stock_code: string ... 7 more fields]

In [0]:
%scala
retail_df = retail_df
  .filter(col("invoice_date").isNotNull && col("customer_id").isNotNull)
  .withColumn("invoice_date", to_date(col("invoice_date"), "yyyyMM"))

// # this is the date to reference for recency factor
val referenceDate = retail_df.agg(max(col("invoice_date"))).first().getAs[java.sql.Date]("max(invoice_date)")

val rfm = retail_df.groupBy("customer_id")
  .agg(
    datediff(lit(referenceDate), max(to_date(col("invoice_date"), "yyyyMM"))).alias("Recency"), // # check for recency value
    countDistinct("invoice_no").alias("Frequency"), // # check for frequency value
    round(sum("total_amount"),2).alias("Monetary") // # check for monetary value
  )
  .orderBy("customer_id")
// display(rfm)
rfm.show(5)

+-----------+-------+---------+--------+
customer_id|Recency|Frequency|Monetary|
+-----------+-------+---------+--------+
 12346| 334| 17| -64.68|
 12347| 0| 8| 5633.32|
 12348| 91| 5| 2019.4|
 12349| 30| 5| 4404.54|
 12350| 303| 1| 334.4|
+-----------+-------+---------+--------+
only showing top 5 rows

retail_df: org.apache.spark.sql.DataFrame = [invoice_no: string, stock_code: string ... 7 more fields]
referenceDate: java.sql.Date = 2011-12-01
rfm: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [customer_id: int, Recency: int ... 2 more fields]

## RFM Segmentation

In [0]:
%scala
val recencyQuantiles = rfm.stat.approxQuantile("Recency", Array(0.2, 0.4, 0.6, 0.8), 0.0)
val frequencyQuantiles = rfm.stat.approxQuantile("Frequency", Array(0.2, 0.4, 0.6, 0.8), 0.0)
val monetaryQuantiles = rfm.stat.approxQuantile("Monetary", Array(0.2, 0.4, 0.6, 0.8), 0.0)

def assignScore(value: Double, thresholds: Array[Double], isRecency: Boolean = false): Int = {
  val score = thresholds.count(value > _)
  if (isRecency) 5 - score else 1 + score
}

val recencyScoreUDF = udf((recency: Double) => assignScore(recency, recencyQuantiles, true))
val frequencyScoreUDF = udf((frequency: Double) => assignScore(frequency, frequencyQuantiles))
val monetaryScoreUDF = udf((monetary: Double) => assignScore(monetary, monetaryQuantiles))

recencyQuantiles: Array[Double] = Array(30.0, 61.0, 183.0, 426.0)
frequencyQuantiles: Array[Double] = Array(1.0, 3.0, 5.0, 10.0)
monetaryQuantiles: Array[Double] = Array(259.17, 584.07, 1185.46, 2858.73)
assignScore: (value: Double, thresholds: Array[Double], isRecency: Boolean)Int
recencyScoreUDF: org.apache.spark.sql.expressions.UserDefinedFunction = SparkUserDefinedFunction($Lambda$10002/1946391750@18eaa535,IntegerType,List(Some(class[value[0]: double])),Some(class[value[0]: int]),None,false,true)
frequencyScoreUDF: org.apache.spark.sql.expressions.UserDefinedFunction = SparkUserDefinedFunction($Lambda$10003/413962720@72c52b9f,IntegerType,List(Some(class[value[0]: double])),Some(class[value[0]: int]),None,false,true)
monetaryScoreUDF: org.apache.spark.sql.expressions.UserDefinedFunction = SparkUserDefinedFunction($Lambda$10004/1184986204@e1a87b6,IntegerType,List(Some(class[value[0]: double])),Some(class[value[0]: int]),None,false,true)

In [0]:
%scala
val scoredRFM = rfm
  .withColumn("recency_score", recencyScoreUDF(col("Recency")))
  .withColumn("frequency_score", frequencyScoreUDF(col("Frequency")))
  .withColumn("monetary_score", monetaryScoreUDF(col("Monetary")))

val combined = scoredRFM.withColumn("segment_code", concat(col("recency_score").cast("string"), col("frequency_score").cast("string")))

// seg map
val segmented = combined.withColumn("segment",
  when(col("segment_code").rlike("[1-2][1-2]"), "Hibernating")
    .when(col("segment_code").rlike("[1-2][3-4]"), "At Risk")
    .when(col("segment_code").rlike("[1-2]5"), "Can't Lose")
    .when(col("segment_code").rlike("3[1-2]"), "About to Sleep")
    .when(col("segment_code") === "33", "Need Attention")
    .when(col("segment_code").rlike("[3-4][4-5]"), "Loyal Customers")
    .when(col("segment_code") === "41", "Promising")
    .when(col("segment_code") === "51", "New Customers")
    .when(col("segment_code").rlike("[4-5][2-3]"), "Potential Loyalists")
    .when(col("segment_code").rlike("5[4-5]"), "Champions")
    .otherwise("Others")
)

val result = segmented.groupBy("segment")
  .agg(
    mean("Recency").alias("Avg_Recency"),
    count("*").alias("Count"),
    mean("Frequency").alias("Avg_Frequency"),
    mean("Monetary").alias("Avg_Monetary")
  )

result.orderBy(desc("Count")).show(false)

+-------------------+------------------+-----+------------------+------------------+
segment |Avg_Recency |Count|Avg_Frequency |Avg_Monetary |
+-------------------+------------------+-----+------------------+------------------+
Hibernating |462.61728395061726|1782 |1.569023569023569 |383.6043265993264 |
Champions |17.608142493638677|1179 |21.674300254452927|9287.408125530126 |
Potential Loyalists|33.16572717023676 |887 |3.3799323562570462|992.928218714767 |
Loyal Customers |96.79439252336448 |642 |11.53582554517134 |3669.0441121495337|
At Risk |372.3203883495146 |515 |5.656310679611651 |1438.5708543689332|
About to Sleep |128.4158653846154 |416 |1.8581730769230769|595.4464903846155 |
Need Attention |129.27748691099475|191 |4.403141361256544 |1355.466387434554 |
New Customers |24.65753424657534 |146 |1.0 |315.69602739726037|
Promising |61.0 |116 |1.0 |331.17698275862085|
Can't Lose |327.61764705882354|68 |19.602941176470587|6568.083235294117 |
+-------------------+------------------+-----+------------------+------------------+

scoredRFM: org.apache.spark.sql.DataFrame = [customer_id: int, Recency: int ... 5 more fields]
combined: org.apache.spark.sql.DataFrame = [customer_id: int, Recency: int ... 6 more fields]
segmented: org.apache.spark.sql.DataFrame = [customer_id: int, Recency: int ... 7 more fields]
result: org.apache.spark.sql.DataFrame = [segment: string, Avg_Recency: double ... 3 more fields]